# DAY 3 — 멀티모달 기반 비정형 데이터 정보화

**AI 적용을 위한 현장 데이터 수집 및 디지털화** · DAY 3 실습

교재 PART 03 (pp.128–178) 의 실습 코드를 코랩에서 바로 돌릴 수 있게 옮긴 것입니다.
설명과 그림은 교재와 학습사이트에 있습니다 — https://build-data.jobability.co.kr

---
### 코랩 쓰는 법 세 가지만

1. **셀 실행** — 코드 칸 왼쪽 ▶ 를 누르거나 `Shift + Enter`.
2. **위에서부터 차례로** — 앞 칸에서 만든 것을 뒤 칸이 씁니다. 건너뛰면 오류가 납니다.
3. **내 사본으로** — 「파일 → 드라이브에 사본 저장」을 먼저 해야 고친 내용이 남습니다.

> **제목의 번호는 교재 절 번호입니다.** 이론만 있는 절은 실행할 것이 없어 빠져 있어서
> 번호가 건너뜁니다(0 → 4 처럼). 빠진 절의 설명은 교재와 학습사이트에 있습니다.

> **실습 데이터는 준비 칸이 자동으로 받아 옵니다.** 교재와 같은 원본이라 설명 글의 숫자가
> 그대로 나옵니다. 내려받기가 막혀 대체 데이터로 진행한 경우에는 행 수·건수처럼 정해 둔 값은
> 같지만 평균·p값처럼 난수에 걸리는 값이 달라집니다 — **읽는 방법이 같으면 맞게 하신 것입니다.**

> **API 키는 왼쪽 🔑(보안 비밀)에 한 번 등록해 두면 편합니다.**
> 이름은 `OPENAI_API_KEY`, 값은 강사가 나눠 준 키(`sk-…`), 그리고 **「노트북 액세스」를 켜면**
> 아래 키 칸이 묻지 않고 그냥 지나갑니다.
> 이 키는 **실습용 임시 공용 키**로 수업이 끝나면 폐기됩니다 — 공유하지 마세요.

## 0. 실습 준비

이 아래 두 칸은 세션을 새로 열 때마다 한 번씩 실행합니다.

In [ ]:
# 그래프에 한글이 네모로 나오지 않게 폰트를 깝니다. 세션마다 한 번만 하면 됩니다.
!apt-get -qq install fonts-nanum > /dev/null 2>&1
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager
import glob, os
cand = glob.glob("/usr/share/fonts/truetype/nanum/NanumGothic*.ttf") + glob.glob("fonts/NanumGothic*.ttf")
if not cand:                      # apt 가 막힌 환경이면 자료 저장소에서 받아 씁니다
    import urllib.request
    os.makedirs("fonts", exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/aebonlee/materials/main/build-data/data/NanumGothic-Regular.ttf",
        "fonts/NanumGothic-Regular.ttf")
    cand = glob.glob("fonts/NanumGothic*.ttf")
if cand:
    font_manager.fontManager.addfont(cand[0])
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=cand[0]).get_name()
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트:", plt.rcParams["font.family"])

In [ ]:
# 실습 데이터 준비
# 1) 왼쪽 폴더 아이콘에 data_day3_docs.zip 을 올려 두었으면 그것을 씁니다.
# 2) 없으면 자료 저장소에서 자동으로 내려받습니다.
# 3) 그것도 막히면 같은 구조의 대체 데이터를 그 자리에서 만들어 씁니다.
import os, zipfile, urllib.request
if not os.path.exists("checklists"):
    if not os.path.exists("data_day3_docs.zip"):
        try:
            print("data_day3_docs.zip 을 자료 저장소에서 받는 중 …")
            urllib.request.urlretrieve("https://raw.githubusercontent.com/aebonlee/materials/main/build-data/data_day3_docs.zip", "data_day3_docs.zip")
        except Exception as e:
            print("  내려받지 못했습니다:", type(e).__name__)
    if os.path.exists("data_day3_docs.zip"):
        zipfile.ZipFile("data_day3_docs.zip").extractall(".")
    else:
        print("대신 같은 구조의 대체 실습 데이터를 만듭니다. 30초쯤 걸립니다.")
        import sys, subprocess
        urllib.request.urlretrieve("https://raw.githubusercontent.com/aebonlee/materials/main/build-data/data/make_day3_data.py", "make_day3_data.py")
        subprocess.run([sys.executable, "make_day3_data.py"], check=True)
print("준비된 폴더:", sorted(d for d in os.listdir(".") if os.path.isdir(d) and not d.startswith(".")))

## 2. 음향 AI와 멀티모달 LLM의 원리

### 음향 신호의 디지털 변환 원리

**[3-1] 음향 파일 하나의 숫자 개수**

```
파일 하나의 숫자 개수 = 샘플레이트 × 길이 = 16,000 × 10 = 160,000개
```

## 4. 실습 A(비전): 결함 사진 자동 라벨링

### 단계 1~2: 환경과 데이터 준비

**[3-2] API 키 입력받기**

노트북의 첫 셀부터 실행합니다. 보안을 위해 API 키는 코드에 직접 노출하지 않고, 실행 시 나타나는 입력창을 통해 입력받습니다. 앞 장에서 설명한 대로 키를 코드에 적지 않고 입력창으로 받습니다.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# API 키 넣는 법 — 두 가지 중 하나
#
# 【권장】 왼쪽 세로 막대의 🔑(열쇠) 아이콘 = 「보안 비밀(Secrets)」에 등록해 두기
#   1. 왼쪽 끝 세로 막대에서 🔑 아이콘을 누릅니다.
#      (폴더📁 아이콘 위·아래에 있습니다. 안 보이면 막대 맨 위 ☰ 를 눌러 펼치세요.)
#   2. 「새 보안 비밀 추가」를 누릅니다.
#   3. 두 칸을 채웁니다.
#
#        이름(Name) : OPENAI_API_KEY     ← 반드시 이 철자 그대로. 대소문자·밑줄까지 같아야 합니다
#        값(Value)  : 강사가 나눠 준 키   ← sk- 로 시작하는 긴 문자열
#
#      이름이 한 글자라도 다르면 노트북이 못 찾습니다(OPENAI-API-KEY, openai_api_key 모두 안 됩니다).
#   4. 왼쪽의 「노트북 액세스」 스위치를 **켭니다.** ← 이걸 안 켜면 등록해도 못 읽습니다
#   5. 이 칸을 실행하면 키를 묻지 않고 그냥 지나갑니다.
#
#   한 번 등록해 두면 새 노트북에서도 계속 쓰입니다. 화면 공유·캡처에도 값이
#   드러나지 않아 가장 안전합니다. (Gemini 키를 쓰신다면 이름을 GEMINI_API_KEY 로)
#
# ⚠ 이 키는 **실습용으로 잠깐 함께 쓰는 키**입니다. 수업이 끝나면 강사가 폐기합니다.
#    - 남에게 공유하거나 개인 프로젝트·깃허브에 올리지 마세요.
#    - 수업이 끝난 뒤 계속 실습하시려면 본인 키를 발급받아 같은 이름으로 바꿔 넣으면 됩니다.
#
# 【임시】 그냥 이 칸을 실행하면 입력창이 뜹니다. 거기에 붙여 넣어도 됩니다.
#   이 방법은 세션이 끊기면 다시 넣어야 합니다.
#
# ※ 어느 쪽이든 키를 코드 칸에 직접 적지 마세요 — 노트북에 그대로 남습니다.
# ─────────────────────────────────────────────────────────────────────────
import os
from getpass import getpass

def _secret(name):
    """코랩 🔑(보안 비밀)에서 꺼낸다. 없거나 접근이 꺼져 있으면 빈 값."""
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

API_KEY = (_secret("OPENAI_API_KEY") or _secret("GEMINI_API_KEY")
           or os.environ.get("OPENAI_API_KEY") or os.environ.get("GEMINI_API_KEY") or "").strip()
if not API_KEY:
    print("🔑 보안 비밀에 등록된 키가 없어 직접 입력받습니다. (위 설명 참고)")
    API_KEY = getpass("API 키 입력 (sk- 로 시작하면 OpenAI, 아니면 Gemini): ").strip()
PROVIDER = "openai" if API_KEY.startswith("sk-") else "gemini"

# 모델을 바꿔야 하면 이 두 줄만 고칩니다.
OPENAI_MODEL = "gpt-4o"                          # 사진을 읽는 모델
OPENAI_STT_MODEL = "gpt-4o-transcribe-diarize"   # 말한 사람을 구분해 받아쓰는 모델

print("사용할 서비스:", PROVIDER, "· 키 길이", len(API_KEY), "자")

셀을 실행하면 입력창이 나타납니다. 발급받은 키를 붙여 넣고 Enter를 누르면 됩니다. 화면에는 아무것도 표시되지 않으며, 입력한 값은 실행 중인 노트북의 메모리에만 남습니다. 다음은 모델을 호출하는 함수입니다. 이 함수 하나를 만들어 두면 이후 모든 호출에서 재사용할 수 있습니다.

**[3-3] Gemini 이미지 호출 함수**

In [ ]:
import base64, json, urllib.request
MODEL = "gemini-3.5-flash"
def gemini_image(prompt, file_path):
    data = base64.b64encode(open(file_path, "rb").read()).decode()
    body = {"contents": [{"parts": [
        {"inline_data": {"mime_type": "image/jpeg", "data": data}},
        {"text": prompt}]}],
        "generationConfig": {"temperature": 0}}
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
    req = urllib.request.Request(url, data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json",
                                          "x-goog-api-key": API_KEY,        # 키는 URL이 아니라 헤더로(노출방지)
                                          "User-Agent": "Mozilla/5.0 (data-pipeline-practice)"})
    resp = json.load(urllib.request.urlopen(req, timeout=180))
    text = resp["candidates"][0]["content"]["parts"][0]["text"]
    return json.loads(text.strip().removeprefix("```json").removesuffix("```").strip("` \n"))
print("호출 준비 완료 — 모델:", MODEL)

if PROVIDER == "openai":
    import mimetypes
    def gemini_image(prompt, file_path):
        """이름과 쓰임새는 위와 같습니다. 보내는 곳만 OpenAI 입니다."""
        b64 = base64.b64encode(open(file_path, "rb").read()).decode()
        mime = mimetypes.guess_type(file_path)[0] or "image/jpeg"
        body = {"model": OPENAI_MODEL, "temperature": 0,
                "messages": [{"role": "user", "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url",
                     "image_url": {"url": f"data:{mime};base64,{b64}"}}]}]}
        req = urllib.request.Request(
            "https://api.openai.com/v1/chat/completions",
            data=json.dumps(body).encode(),
            headers={"Content-Type": "application/json",
                     "Authorization": f"Bearer {API_KEY}"})   # 키는 헤더로(노출 방지)
        text = json.load(urllib.request.urlopen(req, timeout=180))["choices"][0]["message"]["content"]
        return json.loads(text.strip().removeprefix("```json").removesuffix("```").strip("` \n"))
    print("→ OpenAI 로 보냅니다 · 모델:", OPENAI_MODEL)

코드가 길어 보이지만 하는 일은 네 가지입니다. 먼저 사진 파일을 읽어 문자열로 바꿉니다. base64가 이 변환을 담당합니다. 사진은 원래 이진 파일이라 JSON 요청에 그대로 담을 수 없기 때문에, 문자로만 이루어진 형태로 바꿔서 싣습니다. 다음으로 요청 본문을 만듭니다. 사진과 지시문을 하나의 parts 목록에 나란히 담는 구조에 주목하기 바랍니다. 앞 장에서 설명한 "이미지와 텍스트를 같은 자리에서 처리한다"는 원리가 요청 형식에 그대로 드러나 있습니다. temperature를 0으로 둔 것은 응답의 무작위성을 최대한 낮추려는 설정입니다. 그다음이 인증입니다. x-goog-api-key 헤더에 키를 담아 보냅니다. 주소에는 키가 들어가지 않습니다. 마지막으로 응답에서 필요한 부분만 꺼냅니다. 모델이 JSON을 코드 블록 표시와 함께 돌려주는 경우가 있어, 앞뒤의 표시를 떼어 낸 뒤 파싱합니다. 이런 정리 과정은 실무에서 거의 항상 필요합니다.

**[3-4] casting 데이터 내려받기**

이제 데이터를 준비합니다. 이 데이터는 재배포가 금지된 조건으로 공개되어 있어 실습 자료에 포함하지 않았습니다. 대신 노트북에서 원본 배포처에 직접 접속해 내려받습니다.

In [ ]:
import os, glob
try:
    import kagglehub
    path = kagglehub.dataset_download("ravirajsinh45/real-life-industrial-dataset-of-casting-product")
    base = os.path.join(path, "casting_512x512", "casting_512x512")
except Exception as e:
    print("공개 데이터셋을 내려받지 못했습니다:", type(e).__name__)
    print("→ 준비 칸이 만들어 둔 대체 이미지(casting_local)로 진행합니다.")
    base = "casting_local"
def_files = sorted(glob.glob(os.path.join(base, "def_front", "*.jpeg")))
ok_files  = sorted(glob.glob(os.path.join(base, "ok_front", "*.jpeg")))
print("결함(def)", len(def_files), "장 / 정상(ok)", len(ok_files), "장")

실행 결과는 다음과 같습니다.

### 단계 3~4: 육안 확인과 단건 라벨링

**[3-6] 정상 ·결함 사진 나란히 보기**

AI에게 시키기 전에 사람 눈에는 어떻게 보이는지 확인합니다. 이 순서를 지키는 데는 이유가 있습니다. 데이터를 보지 않고 모델부터 붙이면, 결과가 이상하게 나왔을 때 모델이 문제인지 데이터가 문제인지 판단할 수 없습니다.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, f, tag in [(axes[0], ok_files[0], "ok (정상)"), (axes[1], def_files[0], "def (결함)")]:
    ax.imshow(Image.open(f)); ax.set_title(tag); ax.axis("off")
plt.tight_layout(); plt.show()

실행하면 정상 사진과 결함 사진이 나란히 나타납니다. 각자 화면에서 두 사진을 비교해 보기 바랍니다. 확인할 것은 두 가지입니다. 첫째, 두 사진의 차이가 한눈에 들어오는지 봅니다. 둘째, 만약 라벨을 가리고 보여 준다면 어느 쪽이 결함인지 자신 있게 고를 수 있을지 생각해 봅니다. 주조품 표면의 결함 중에는 크고 분명한 것도 있지만, 미세한 기공처럼 확대해서 봐야 겨우 보이는 것도 있습니다. 사람 눈으로도 확신하기 어려운 사진이 섞여 있다는 사실을 미리 알고 있어야, 뒤에서 AI가 틀렸을 때 그 결과를 제대로 해석할 수 있습니다. AI 모델을 적용하기 전, 데이터를 육안으로 직접 확인하여 분석의 기준선을 설정해야 합니다. 이는 결과가 좋지 않을 때 문제의 원인이 모델인지 데이터 자체인지를 판단하기 위한 필수 절차입니다. 히지만, 이진 파일을 메모장으로 열면 깨진 기호만 나옵니다. 값의 나열이지 글자의 나열이 아니기 때문입니다. 문제는 API 요청의 본문이 대개 글자로 이루어진 JSON이라는 데 있습니다. 글자만 담을 수 있는 그릇에 글자가 아닌 것을 넣어야 하는 상황입니다. base64는 이 문제를 푸는 약속입니다. 이진 값을 정해진 64가지 글자만 써서 표현하는 방식이라, 어떤 파일이든 긴 영문·숫자 문자열로 바뀝니다. 압축이 아니라 표현 방식의 변환이므로 크기는 오히려 3분의 4배쯤 늘어납니다. 사진을 API로 보낼 때 요청 본문이 유난히 길어지는 이유가 여기에 있습니다.

**단계 4. 단건 라벨링 요청하기**

**[3-7] 자연어 지시: 결함 판정 라벨링**

```
이 주조품 사진에 표면 결함이 있으면 def, 없으면 ok로 판정하고
근거를 한 문장으로 JSON에 담아줘.
```

**[3-8] 프롬프트: 주조품 표면 결함 판정**

```
이 사진은 엔진 냉각수 펌프 임펠러 주조품의 입고 검사 사진입니다.
표면 결함(기공, 균열, 미성형, 버, 흠집 등)이 있는지 판정하세요.
{"라벨": "ok|def", "근거": "한 문장"} JSON만 출력하세요. 결함이 있으면 def, 없으면 ok.
프롬프트를 뜯어보면 네 가지가 들어 있습니다.
```

**[3-9] 사진 한 장 라벨링**

첫 줄은 맥락입니다. 이것이 무슨 사진인지 알려 줍니다. 같은 사진이라도 "예술 작품 사진"이라고 알려 주면 다른 관점으로 볼 것입니다. 둘째 줄은 판정 기준입니다. 무엇을 결함으로 볼 것인지 유형을 나열했습니다. 이 나열이 곧 검사 기준서 역할을 합니다. 우리 회사에 적용한다면 이 자리에 실제 검사 기준의 항목이 들어가야 합니다. 셋째 줄 앞부분은 출력 형식입니다. 라벨과 근거 두 필드를 가진 JSON을 요구했습니다. 형식을 정해 두어야 60건을 자동으로 모아 채점할 수 있습니다. 셋째 줄 뒷부분은 값의 정의입니다. def와 ok가 각각 무엇을 뜻하는지 명시했습니다. 형식만 주고 값의 의미를 빠뜨리면 모델이 "결함"이나 "불량" 같은 다른 표기를 쓸 수 있고, 그러면 채점 코드가 전부 어긋납니다. 프롬프트가 검사 기준서 역할을 한다면, 검사 기준서에 하는 관리를 프롬프트에도 해야 합니다. 문구를 바꾸면 판정 결과가 달라지므로, 어떤 문구로 몇 장을 라벨링했는지가 남아 있어야 나중에 결과를 비교할 수 있습니다. 프롬프트를 고칠 때는 이전 문구를 지우지 말고 날짜와 함께 보관하고, 어느 문구로 만든 라벨인지를 결과에 함께 기록해 두기 바랍니다. 몇 줄짜리 문구라고 가볍게 다루면, 두 달 뒤 정확도가 달라졌을 때 프롬프트가 바뀐 탓인지 데이터가 바뀐 탓인지 가릴 수 없게 됩니다.

In [ ]:
PROMPT = '''이 사진은 엔진 냉각수 펌프 임펠러 주조품의 입고 검사 사진입니다.
표면 결함(기공, 균열, 미성형, 버, 흠집 등)이 있는지 판정하세요.
{"라벨": "ok|def", "근거": "한 문장"} JSON만 출력하세요. 결함이 있으면 def, 없으면 ok.'''
r = gemini_image(PROMPT, def_files[0])
print(r)

실행하면 라벨과 근거 두 필드를 담은 결과가 한 건 출력됩니다. 라벨 자리에는 ok 또는 def가, 근거 자리에는 그렇게 판정한 이유를 적은 한 문장이 들어 있습니다. 여기서 결과 한 건을 보고 판단을 내리지 않도록 주의해야 합니다. 한 장이 맞았다고 해서 이 프롬프트가 쓸 만하다는 뜻이 아니고, 한 장이 틀렸다고 해서 못 쓴다는 뜻도 아닙니다. 지금 확인한 것은 응답이 우리가 요구한 형식으로 돌아온다는 사실뿐입니다. 형식이 맞는지부터 확인하고 규모를 늘리는 것이 순서입니다.

### 단계 5~6: 60장 일괄 라벨링과 혼동 행렬

**[3-10] 결함 30장 ·정상 30장 일괄 라벨링**

결함 30장과 정상 30장, 총 60장을 대상으로 라벨링을 진행합니다. 코드에서 세 가지를 눈여겨보기 바랍니다. 먼저 true 필드입니다. 반복문이 폴더별로 돌면서 그 폴더의 이름을 정답으로 함께 기록합니다. 이 한 줄 덕분에 채점 준비가 라벨링과 동시에 끝납니다. 다음은 try 구문입니다. 60번 호출하는 동안 한 건쯤은 실패할 수 있습니다. 실패한 건에서 전체가 멈추면 앞의 성공분까지 날아가므로, 실패는 None으로 기록하고 넘어가게 했습니다. 마지막은 time.sleep(0.4)입니다. 호출과 호출 사이에 0.4초를 쉽니다. 앞 장에서 설명한 분당 요청 한도 대응입니다. 이 대기 때문에 60장을 처리하는 데 4~5분이 걸립니다. 실행해 두고 기다리기 바랍니다.

In [ ]:
import time
results = []
for true, files in [("def", def_files[:30]), ("ok", ok_files[:30])]:
    for f in files:
        try:
            r = gemini_image(PROMPT, f)
            pred = r.get("라벨")
        except Exception:
            r, pred = {}, None
        results.append({"file": os.path.basename(f), "true": true,
                        "pred": pred, "근거": r.get("근거", "")})
        time.sleep(0.4)
print("라벨링 완료:", len(results), "장")

60장이라는 규모는 임의로 정한 것이 아닙니다. 정상과 결함을 같은 수로 맞춘 것은 한쪽으로 치우친 표본에서 정확도가 왜곡되는 것을 막기 위해서입니다. 결함 사진만 60장 넣으면 모든 사진을 결함이라고 답하는 모델도 정확도 100%를 받습니다. 실제 입고 검사에서는 정상품이 훨씬 많으므로, 현장 비율 그대로 표본을 뽑으면 "전부 정상"이라고 답하는 것만으로도 높은 정확도가 나옵니다. 검증용 표본은 현장 비율이 아니라 판정 능력을 재기 좋은 비율로 구성해야 합니다. 근거 필드를 함께 저장하는 것도 의도가 있습니다. 라벨만 모으면 몇 건을 틀렸는지까지만 알 수 있고, 왜 틀렸는지는 알 수 없습니다. 근거를 함께 남겨 두면 뒤에서 오분류를 재검할 때 모델이 무엇을 보고 그렇게 판정했는지 확인할 수 있습니다. 이 한 필드가 다음 단계의 대책을 정하는 근거가 됩니다.

**[3-11] 혼동 행렬로 채점하기**

라벨링이 끝났으니 채점합니다.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
y_true = [r["true"] for r in results]
y_pred = [r["pred"] or "?" for r in results]
cm = confusion_matrix(y_true, y_pred, labels=["def", "ok"])
ConfusionMatrixDisplay(cm, display_labels=["def(결함)", "ok(정상)"]).plot(cmap="Blues")
plt.title("LLM 자동 라벨링 혼동 행렬")
plt.show()
acc = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)
print(f"정확도 {acc:.1%} | 미탐(결함→ok) {cm[0,1]}건 | 오탐(정상→def) {cm[1,0]}건")

실행하면 혼동 행렬 그래프가 나타나고 이어서 요약 수치가 출력됩니다.

### 단계 7: 오분류 재검과 검토 규칙 수립

**[3-13] 오분류 사진 재검**

왜 놓쳤는지 확인해야 다음 대책이 나옵니다. 틀린 건을 다시 열어 봅니다.

In [ ]:
miss = [r for r in results if r["true"] != r["pred"]]
print(f"오분류 {len(miss)}건")
if miss:
    fig, axes = plt.subplots(1, min(3, len(miss)), figsize=(11, 4))
    axes = [axes] if len(miss) == 1 else list(axes)
    lookup = {os.path.basename(f): f for f in def_files[:30] + ok_files[:30]}
    for ax, m in zip(axes, miss[:3]):
        ax.imshow(Image.open(lookup[m["file"]]))
        ax.set_title(f"{m['true']} → {m['pred']}", fontsize=10); ax.axis("off")
    plt.tight_layout(); plt.show()
    for m in miss[:3]:
        print(f"- {m['file']}: {m['근거'][:60]}")

틀린 사진 세 장과 그때 모델이 적은 근거가 함께 출력됩니다. 사진은 각자 화면에서 확인하기 바랍니다. 여기서 근거 텍스트를 읽어 보면 중요한 사실이 드러납니다. 미탐이 난 건들의 근거는 대체로 "결함이 발견되지 않고 표면 상태가 양호하다"는 취지의 문장입니다. 모델이 결함을 보고도 무시한 것이 아니라, 애초에 보지 못한 것입니다. 이 구분이 중요한 이유는 대책이 달라지기 때문입니다. 모델이 결함을 보고도 기준을 느슨하게 적용한 것이라면 프롬프트에서 기준을 강화하면 됩니다. 그런데 미세한 기공을 아예 보지 못하는 것이라면 프롬프트를 고쳐도 크게 나아지지 않습니다. 해상도를 높이거나, 결함이 잘 보이는 각도로 다시 촬영하거나, 확대 이미지를 함께 보내는 쪽으로 접근해야 합니다. 당장 할 수 있는 대책은 따로 있습니다. AI 라벨을 그대로 확정하지 않고 사람이 다시 보는 단계를 붙이는 것입니다. 다만 전량을 다시 보면 AI를 쓴 의미가 없으므로, 어디를 다시 볼지 정해야 합니다. 여기서 앞의 혼동 행렬이 근거가 됩니다. 오류가 미탐에 쏠려 있으므로 검토 규칙도 방향에 따라 달라야 합니다. def로 판정된 건은 오탐이 1건뿐이었으니 그대로 반품 절차로 보내도 위험이 작습니다. 반면 ok로 판정된 건에는 놓친 결함이 섞여 있을 가능성이 있으므로 표본을 뽑아 다시 봐야 합니다. 판정 방향에 따라 다른 규칙을 적용하는 이런 설계를 비대칭 검토 규칙이라고 부릅니다. 이 규칙을 흐름으로 그려 보면 사람이 개입하는 지점이 분명해집니다. AI 판정이 def인 건은 반품 검토 절차로 바로 보냅니다. 오탐이 섞여 있더라도 반품 처리 과정에서 자연스럽게 걸러지기 때문입니다. ok인 건은 일부를 표본으로 뽑아 사람이 다시 본 뒤에 통과시키고, 표본 재검에서 미탐이 발견되면 해당 묶음 전체를 다시 보는 식으로 안전장치를 겹칩니다. 전량을 사람이 보던 구조가, 사람의 눈이 가장 위험한 방향에 집중되는 구조로 바뀐 것입니다.

**[3-14] 사람 검토 규칙 설계 양식**

아래를 복사해 **직접 채워 보세요.** 실행하는 칸이 아니라 오늘의 산출물입니다.

```
[대상 업무]
(예: 임펠러 주조품 입고 검사)
[더 치명적인 오류]  미탐 / 오탐 중 택1
그렇게 판단한 이유:
[def 판정 건의 처리]
자동 확정 / 전량 재검 / 표본 재검(   %)
이유:
[ok 판정 건의 처리]
자동 확정 / 전량 재검 / 표본 재검(   %)
이유:
[재검에서 오류가 발견되면]
(예: 해당 배치 전량 재검, 프롬프트 기준 보완, 촬영 방식 변경)
[규칙을 다시 검토할 시점]
(예: 누적 500건마다, 납품처가 바뀔 때)
```

## 5. 실습 B(음향): 가동음 정상·이상 분류

### 단계 1~2: 데이터 준비와 파형 확인

**[3-15] 음향 데이터 압축 해제**

Colab 파일 탭에 data_day3_audio.zip 파일을 업로드한 뒤, 아래 셀을 실행하여 압축을 해제합니다.

In [ ]:
import os, zipfile
if not os.path.exists("audio/normal"):
    if os.path.exists("data_day3_audio.zip"):
        zipfile.ZipFile("data_day3_audio.zip").extractall(".")
    else:
        # 설비 가동음은 배포 조건 때문에 교재 저장소에도 없다 — 같은 성질의 대체 음원을 만든다.
        print("설비 가동음을 만듭니다. 30초쯤 걸립니다.")
        import sys, subprocess, urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/aebonlee/materials/main/build-data/data/make_day3_data.py",
            "make_day3_data.py")
        subprocess.run([sys.executable, "make_day3_data.py"], check=True)
import glob
n_files = sorted(glob.glob("audio/normal/*.wav"))
a_files = sorted(glob.glob("audio/abnormal/*.wav"))
print("정상", len(n_files), "개 / 이상", len(a_files), "개")

실행 결과는 다음과 같습니다.

**[3-17] librosa와 한글 폰트 준비**

앞 실습의 casting 데이터와 같은 구조입니다. normal과 abnormal 두 폴더에 파일이 나뉘어 있고, 폴더 이름이 곧 라벨입니다. 소리는 사진과 달리 사람이 들어 봐도 라벨을 붙이기 어려우므로, 녹음 시점의 장비 상태를 그대로 기록해 두는 것이 유일한 라벨링 방법입니다. 이 라벨 방식에는 조건이 하나 붙어 있다는 점도 봐 두기 바랍니다. 녹음할 때 장비 상태를 확실히 알고 있어야 폴더 구분이 정답 구실을 합니다. 고장인지 아닌지 애매한 상태에서 녹음한 파일이 이상 폴더에 섞이면, 그 라벨의 오염은 뒤의 어떤 단계에서도 걸러지지 않습니다. 나중에 우리 장비의 소리를 모을 때도 마찬가지입니다. 녹음 당시의 장비 상태를 정비 기록으로 확인할 수 있는 파일만 라벨 데이터로 쓰고, 상태가 불확실한 파일은 따로 보관하는 규칙이 필요합니다. 다음은 음향 처리 라이브러리를 준비하는 셀입니다. 그래프에 한글 제목을 넣기 위해 실습 자료에 포함된 폰트를 등록합니다.

In [ ]:
import librosa, librosa.display
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
import glob
_font = (glob.glob("fonts/NanumGothic*.ttf")
         + glob.glob("/usr/share/fonts/truetype/nanum/NanumGothic*.ttf"))
if _font:
    font_manager.fontManager.addfont(_font[0])
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=_font[0]).get_name()
else:
    print("한글 폰트를 찾지 못했습니다 — 맨 위 「한글 폰트」 칸을 먼저 실행하세요.")
plt.rcParams["axes.unicode_minus"] = False
print("librosa", librosa.__version__)

**단계 2. 파형 확인하기**

**[3-19] 자연어 지시: 파형 나란히 그리기**

```
정상과 이상 펌프 wav를 하나씩 읽어 파형을 나란히 그려줘.
```

**[3-20] 정상 ·이상 파형 그리기**

In [ ]:
yn, sr = librosa.load(n_files[0], sr=None)
ya, _  = librosa.load(a_files[0], sr=None)
print(f"샘플레이트 {sr} Hz, 길이 {len(yn)/sr:.0f}초")
fig, axes = plt.subplots(2, 1, figsize=(10, 3.6), sharex=True)
axes[0].plot(np.arange(len(yn))/sr, yn, lw=0.3); axes[0].set_title("정상 펌프 — 파형")
axes[1].plot(np.arange(len(ya))/sr, ya, lw=0.3, color="tab:orange"); axes[1].set_title("이상 펌프 — 파형")
axes[1].set_xlabel("시간(초)")
plt.tight_layout(); plt.show()

### 단계 3: 스펙트로그램 시각 분석

**단계 3. 스펙트로그램과 평균 스펙트럼**

**[3-22] 자연어 지시: 스펙트로그램과 평균 스펙트럼**

```
두 소리의 멜 스펙트로그램을 나란히 그리고, 각 20개 파일의 평균 스펙트럼을
겹쳐 그려서 차이가 나는 주파수 대역을 찾아줘.
```

**[3-23] 멜 스펙트로그램 그리기**

먼저 멜 스펙트로그램입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, (y, tag) in zip(axes, [(yn, "정상"), (ya, "이상")]):
    M = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=96)
    img = librosa.display.specshow(librosa.power_to_db(M, ref=np.max), sr=sr,
                                   x_axis="time", y_axis="mel", ax=ax, cmap="magma", vmin=-70, vmax=0)
    ax.set_title(f"{tag} 펌프 — 멜 스펙트로그램"); ax.set_xlabel("시간(초)")
fig.colorbar(img, ax=axes, format="%+2.0f dB", shrink=0.85)
plt.show()

**[3-24] 평균 스펙트럼 비교**

In [ ]:
def avg_spec(files):
    return np.mean([np.abs(librosa.stft(librosa.load(f, sr=None)[0], n_fft=2048)).mean(axis=1)
                    for f in files], axis=0)
n_avg, a_avg = avg_spec(n_files[:20]), avg_spec(a_files[:20])
freqs = np.linspace(0, sr/2, len(n_avg))
plt.figure(figsize=(10, 3.2))
plt.plot(freqs, 20*np.log10(n_avg), label="정상(20개 평균)")
plt.plot(freqs, 20*np.log10(a_avg), label="이상(20개 평균)")
plt.axvspan(2000, 4000, alpha=0.12, color="red")
plt.xlabel("주파수(Hz)"); plt.ylabel("크기(dB)"); plt.xlim(0, 8000)
plt.title("평균 스펙트럼 — 이상음은 2~4kHz 대역이 약 +6dB 높다")
plt.legend(); plt.tight_layout(); plt.show()

두 셀의 결과를 함께 놓으면 다음과 같습니다. 그래프는 실습 노트북에서 생성한 자체 제작 그림입니다. 위쪽 두 그림이 멜 스펙트로그램입니다. 왼쪽이 정상, 오른쪽이 이상입니다. 왼쪽에서는 512Hz 부근과 1,024Hz 부근에 밝은 가로 줄무늬가 또렷하게 이어집니다. 앞 장에서 설명한 배음입니다. 펌프가 일정한 속도로 규칙적으로 돌고 있다는 표시입니다. 오른쪽을 보면 줄무늬가 흐려지고 화면 전체가 붉게 밝아져 있습니다. 특정 주파수가 아니라 넓은 대역에 소리가 퍼졌다는 뜻입니다. 정상은 또렷한 줄무늬, 이상은 광대역 잡음이라는 대비가 그림에서 그대로 확인됩니다. 아래쪽 그림은 각 20개 파일의 평균 스펙트럼을 겹쳐 그린 것입니다. 개별 파일의 우연한 변동을 평균으로 지우면 차이가 더 분명해집니다. 두 선은 낮은 주파수에서는 거의 붙어 있다가, 2,000Hz를 넘어서면서 벌어지기 시작합니다. 붉게 칠한 2~4kHz 구간에서 이상음 쪽이 약 6dB 높습니다. 왜 하필 2~4kHz일까요. 회전 기계에서 마모나 미세한 충격이 생기면 짧고 날카로운 소리가 반복해서 발생합니다. 짧은 소리일수록 높은 주파수 성분을 많이 포함하므로, 이상의 흔적은 회전 자체가 만드는 낮은 주파수보다 그 위쪽 대역에 나타납니다. 김 반장이 이상음을 두고 "쇳소리가 섞인다"고 표현했다면, 그 표현이 가리키는 것이 대체로 이 대역입니다.

### 단계 4: MFCC 특징 추출

**단계 4. MFCC 특징 추출**

**[3-25] 자연어 지시: MFCC 특징 행렬 만들기**

```
모든 wav에서 MFCC 13개의 평균과 표준편차를 뽑아 특징 행렬 X와 라벨 y를 만들어줘.
```

**[3-26] MFCC 특징 추출**

In [ ]:
X, y = [], []
for label, files in [(0, n_files), (1, a_files)]:
    for f in files:
        sig, _ = librosa.load(f, sr=None)
        m = librosa.feature.mfcc(y=sig, sr=sr, n_mfcc=13)
        X.append(np.r_[m.mean(axis=1), m.std(axis=1)])
        y.append(label)
X, y = np.array(X), np.array(y)
print("특징 행렬:", X.shape, "(파일 200개 × 특징 26개)")

파일 하나가 16만 개의 숫자에서 26개의 숫자로 줄었습니다. 6천분의 1 규모입니다.

### 단계 5: 분류와 혼동 행렬

**단계 5. 정상·이상 자동 분류**

**[3-28] 자연어 지시: 분류와 혼동 행렬**

```
특징을 7:3으로 나눠 로지스틱 회귀로 학습하고, 테스트 정확도와 혼동 행렬을 보여줘.
```

**[3-29] 로지스틱 회귀 학습과 평가**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
pred = clf.predict(Xte)
print(f"테스트 정확도: {accuracy_score(yte, pred):.3f} (테스트 {len(yte)}개)")

코드의 세 가지 설정을 짚고 넘어가겠습니다.

**[3-31] 혼동 행렬 그리기**

test_size=0.3은 200개 중 30%인 60개를 테스트용으로 떼어 둔다는 뜻입니다. 학습에 쓴 데이터로 성능을 재면 시험 문제를 미리 보고 시험을 치는 것과 같으므로, 학습에 쓰지 않은 데이터로 재야 합니다. stratify=y는 나눌 때 정상과 이상의 비율을 유지하라는 지시입니다. 이 옵션 덕분에 테스트 60개가 정상 30개, 이상 30개로 정확히 반씩 나뉩니다. random_state=42는 나누는 방식을 고정하는 값입니다. 이 값을 지정하지 않으면 실행할 때마다 다른 조합이 뽑혀 정확도가 달라집니다. 결과를 비교하려면 조건을 고정해야 합니다. 분류기로 로지스틱 회귀를 고른 데도 이유가 있습니다. 구조가 단순해 학습이 빠르고, 어떤 특징이 판정에 얼마나 기여했는지 확인하기 쉽습니다. 성능을 최대로 끌어올리는 것이 목적이라면 더 복잡한 모델을 쓸 수 있지만, 지금 확인하려는 것은 "이 특징에 판별 가능한 정보가 들어 있는가"입니다. 단순한 모델로 낮은 성능이 나왔을 때 그것이 모델 탓인지 특징 탓인지 구분하려면, 먼저 단순한 것부터 시도해 보는 편이 낫습니다. 정확도는 0.683입니다. 60개 중 41개를 맞혔습니다. 사진 라벨링이 88~90%였던 것과 비교하면 크게 낮습니다. 왜 그런지 보기 전에 어느 방향으로 틀렸는지부터 확인합니다.

In [ ]:
cm = confusion_matrix(yte, pred)
ConfusionMatrixDisplay(cm, display_labels=["정상", "이상"]).plot(cmap="Blues")
plt.title("혼동 행렬 — 어떤 쪽을 얼마나 틀리는가")
plt.show()
print("이상을 정상으로 놓친 건수(미탐):", cm[1, 0], "/ 정상을 이상으로 오인(오탐):", cm[0, 1])

혼동 행렬 그래프가 함께 나타납니다. 표로 옮기면 다음과 같습니다.

## 6. 실습 C(문서·음성): 추출·검증·적재

### 점검표 사진의 JSON 구조화 추출

**[3-33] 문서 번들 압축 해제**

data_day3_docs.zip 파일을 Colab에 업로드한 후 압축을 해제합니다.

In [ ]:
import os, zipfile
if not os.path.exists("checklists"):
    if os.path.exists("data_day3_docs.zip"):
        zipfile.ZipFile("data_day3_docs.zip").extractall(".")
    else:
        raise SystemExit("데이터가 없습니다 — 위쪽 「실습 데이터 준비」 칸을 먼저 실행하세요.")
print(sorted(d for d in os.listdir(".") if os.path.isdir(d) and not d.startswith(".")))

폴더 이름이 곧 데이터의 종류입니다. checklists에 점검표 사진과 정답 JSON이, audio에 상담 녹음과 대본이, drawings에 도면이, parts에 부품 마스터가 들어 있습니다. 파일명이 모두 영문인 점도 확인해 두기 바랍니다. 실습 환경에 따라 한글 파일명에서 인코딩 문제가 생기는 경우가 있어, 배포 자료의 파일명은 전부 영문으로 통일했습니다. 파일 안의 내용과 열 이름은 한글 그대로입니다. API 키는 앞 실습과 같은 방식으로 입력받습니다.

**[3-35] 문서 파이프라인 API 키 입력**

In [ ]:
# 앞의 「API 키」 칸에서 정한 값을 그대로 씁니다 — 다시 넣지 않아도 됩니다.
try:
    API_KEY, PROVIDER
except NameError:
    raise SystemExit("위쪽 「API 키」 칸을 먼저 실행하세요. (실습을 건너뛰고 오셨다면 거기부터)")
print("사용할 서비스:", PROVIDER)

호출 함수는 앞 실습보다 조금 확장했습니다. 이번에는 이미지뿐 아니라 음성도 보내야 하고, JSON이 아니라 일반 텍스트를 받는 경우도 있기 때문입니다.

**[3-36] Gemini 파일 ·프롬프트 호출 함수**

In [ ]:
import base64, json, urllib.request
MODEL = "gemini-3.5-flash"
def gemini(prompt, file_path=None, mime=None):
    """파일(이미지·음성)과 프롬프트를 Gemini에 보내고 텍스트 응답을 받는다."""
    parts = []
    if file_path:
        data = base64.b64encode(open(file_path, "rb").read()).decode()
        parts.append({"inline_data": {"mime_type": mime, "data": data}})
    parts.append({"text": prompt})
    body = {"contents": [{"parts": parts}], "generationConfig": {"temperature": 0}}
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
    req = urllib.request.Request(url, data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json",
                                          "x-goog-api-key": API_KEY,        # 키는 URL이 아니라 헤더로(노출방지)
                                          "User-Agent": "Mozilla/5.0 (data-pipeline-practice)"})
    resp = json.load(urllib.request.urlopen(req, timeout=300))
    return resp["candidates"][0]["content"]["parts"][0]["text"]
def as_json(text):
    """응답에서 JSON만 꺼낸다(코드 펜스 제거)."""
    return json.loads(text.strip().removeprefix("```json").removesuffix("```").strip("` \n"))

if PROVIDER == "openai":
    import os, uuid, mimetypes
    def gemini(prompt, file_path=None, mime=None):
        """이름과 쓰임새는 위와 같습니다. 보내는 곳만 OpenAI 입니다.
        음성은 대화 창구가 아니라 전사 전용 창구로 갑니다 — 창구가 다릅니다."""
        if file_path and (mime or "").startswith("audio"):
            bd = uuid.uuid4().hex
            payload = (f"--{bd}\r\nContent-Disposition: form-data; name=\"model\"\r\n\r\n"
                       f"{OPENAI_STT_MODEL}\r\n").encode()
            payload += (f"--{bd}\r\nContent-Disposition: form-data; name=\"file\"; "
                        f"filename=\"{os.path.basename(file_path)}\"\r\n"
                        f"Content-Type: {mime}\r\n\r\n").encode()
            payload += open(file_path, "rb").read() + b"\r\n" + f"--{bd}--\r\n".encode()
            req = urllib.request.Request(
                "https://api.openai.com/v1/audio/transcriptions", data=payload,
                headers={"Authorization": f"Bearer {API_KEY}",
                         "Content-Type": f"multipart/form-data; boundary={bd}"})
            return json.load(urllib.request.urlopen(req, timeout=300)).get("text", "")
        content = [{"type": "text", "text": prompt}]
        if file_path:
            b64 = base64.b64encode(open(file_path, "rb").read()).decode()
            m = mime or mimetypes.guess_type(file_path)[0] or "image/png"
            content.append({"type": "image_url",
                            "image_url": {"url": f"data:{m};base64,{b64}"}})
        body = {"model": OPENAI_MODEL, "temperature": 0,
                "messages": [{"role": "user", "content": content}]}
        req = urllib.request.Request(
            "https://api.openai.com/v1/chat/completions",
            data=json.dumps(body).encode(),
            headers={"Content-Type": "application/json",
                     "Authorization": f"Bearer {API_KEY}"})
        return json.load(urllib.request.urlopen(req, timeout=300))["choices"][0]["message"]["content"]
    print("→ OpenAI 로 보냅니다 · 사진:", OPENAI_MODEL, "· 음성:", OPENAI_STT_MODEL)

바뀐 부분은 두 가지입니다. mime 인자를 받아 이미지든 음성이든 같은 함수로 보낼 수 있게 했고, JSON 파싱을 as_json으로 분리해 텍스트 응답이 필요한 경우에는 파싱을 건너뛸 수 있게 했습니다. 같은 함수 하나로 사진과 음성을 모두 처리한다는 점을 눈여겨보기 바랍니다. 앞 장에서 설명한 멀티모달의 실무적 이점이 이 코드에 그대로 드러납니다. 예전이라면 OCR 엔진과 음성 인식 엔진을 각각 붙여야 했을 작업입니다.

**[3-37] 점검표 사진 확인**

먼저 사진을 눈으로 확인합니다.

In [ ]:
from IPython.display import Image, display
display(Image("checklists/daily_001.png", width=520))

화면에 나타나는 점검표는 이 과정 1일차의 「데이터 일관성 붕괴 사례와 예방 방안」에서 살펴본 그 서식과 같습니다. 상단에 장비ID·점검일자·점검자가 있고, 그 아래 점검 항목이 나열되어 있으며, 각 항목마다 양호와 불량을 표시하는 칸과 비고란이 있습니다. 손글씨로 채워져 있고 일부 칸은 비어 있습니다. 이제 이 사진에서 값을 뽑아냅니다.

**[3-38] 자연어 지시: 점검표 항목 추출**

```
이 점검표 사진에서 장비ID·점검일자·점검자와 항목별 판정(양호/불량)·비고를
JSON으로 추출해줘.
```

**[3-39] 프롬프트: 일일점검표 JSON 추출**

```
이 사진은 굴착기 일일점검표입니다. 다음 JSON 형식으로 내용을 추출하세요.
{"장비ID": "", "점검일자": "YYYY-MM-DD", "점검자": "",
"항목": {"<항목명>": {"판정": "양호|불량", "비고": ""}}, "특이사항": ""}
항목명은 사진에 적힌 그대로, JSON만 출력하세요.
프롬프트에서 두 가지를 짚어 두겠습니다.
```

**[3-40] 점검표 한 장 추출**

먼저 날짜 형식을 YYYY-MM-DD로 지정했습니다. 점검표에는 "24. 1. 14."처럼 적혀 있을 수 있는데, 이대로 받으면 나중에 날짜로 정렬하거나 기간을 계산할 수 없습니다. 추출 단계에서 형식을 통일해 두면 뒤에서 손볼 일이 줄어듭니다. 다음으로 "항목명은 사진에 적힌 그대로"라고 지시했습니다. 항목명을 모델이 임의로 다듬으면 정답지와 대조할 때 이름이 어긋나 채점이 되지 않습니다. 채점을 하려면 키가 정확히 일치해야 한다는 요구가 프롬프트에 반영된 것입니다.

In [ ]:
PROMPT_DAILY = '''이 사진은 굴착기 일일점검표입니다. 다음 JSON 형식으로 내용을 추출하세요.
{"장비ID": "", "점검일자": "YYYY-MM-DD", "점검자": "",
 "항목": {"<항목명>": {"판정": "양호|불량", "비고": ""}}, "특이사항": ""}
항목명은 사진에 적힌 그대로, JSON만 출력하세요.'''
ext = as_json(gemini(PROMPT_DAILY, "checklists/daily_001.png", "image/png"))
print(json.dumps(ext, ensure_ascii=False, indent=1)[:400], "'''")

실행하면 프롬프트에서 지정한 구조 그대로 JSON이 출력됩니다. 최상위에 장비ID·점검일자·점검자·항목·특이사항이 있고, 항목 아래에 점검 항목별로 판정과 비고가 들어갑니다. 종이 한 장이 프로그램에서 다룰 수 있는 데이터 구조가 되었습니다.

**[3-41] 정답지와 대조해 채점하기**

여기서 멈추면 안 됩니다. 출력된 JSON을 눈으로 훑어보고 "잘 뽑혔네"라고 넘어가는 것이 가장 흔한 실수입니다. 항목이 열 개가 넘으면 눈으로는 어긋난 자리를 찾지 못하고, 몇 장만 넘어가도 확인 자체를 포기하게 됩니다. 실습 자료에는 이 점검표들의 정답 JSON이 함께 들어 있습니다. 정답과 대조해 정확도를 숫자로 재겠습니다.

In [ ]:
gt_all = {p["파일"]: p for p in json.load(open("checklists/checklists_ground_truth.json"))["사진"]}
def score_daily(file_name, ext):
    gt = gt_all[file_name]
    hit = sum(1 for k, v in gt["항목"].items()
              if (ext.get("항목") or {}).get(k, {}).get("판정") == v["판정"])
    head = [ext.get("장비ID") == gt["장비ID"], ext.get("점검일자") == gt["점검일자"],
            ext.get("점검자") == gt["점검자"]]
    return hit, len(gt["항목"]), sum(head)
hit, total, head = score_daily("daily_001.png", ext)
print(f"항목 판정 {hit}/{total} 일치, 헤더(장비·일자·점검자) {head}/3 일치")

채점 방식은 두 갈래입니다. 항목별 판정이 정답과 같은지 세고, 헤더 세 항목이 각각 맞았는지 확인합니다. 실행하면 몇 개 중 몇 개가 일치했는지가 두 숫자로 출력됩니다. 채점 함수를 이렇게 나눈 데는 이유가 있습니다. 헤더가 틀리는 것과 항목 판정이 틀리는 것은 성격이 다른 오류입니다. 헤더가 틀리면 그 점검표 전체가 엉뚱한 장비의 기록으로 들어가므로 한 건이 통째로 못 쓰게 됩니다. 항목 판정 하나가 틀리는 것은 그 항목만 잘못되는 것입니다. 오류의 성격이 다르면 대책도 달라야 하므로 채점부터 나눠서 재야 합니다.

**[3-42] 점검표 5장 일괄 채점**

한 장이 되면 여러 장은 반복문으로 감싸면 됩니다. 여기서부터가 파이프라인입니다.

In [ ]:
import time
rows = []
for f in ["daily_001.png", "daily_002.png", "daily_003.png", "daily_004.png", "daily_005.png"]:
    e = as_json(gemini(PROMPT_DAILY, f"checklists/{f}", "image/png"))
    hit, total, head = score_daily(f, e)
    rows.append((f, hit, total, head))
    print(f"{f}: 항목 {hit}/{total}, 헤더 {head}/3")
    time.sleep(1)
acc = sum(r[1] for r in rows) / sum(r[2] for r in rows)
print(f"\n항목 판정 정확도: {acc:.1%}")

파일 이름을 목록으로 두고 하나씩 돌면서 추출과 채점을 이어 붙였습니다. 사진마다 결과가 한 줄씩 출력되고 마지막에 전체 정확도가 나옵니다. 호출 사이에 1초를 쉬는 것은 앞 실습과 같은 이유입니다. 여기까지 오면 파이프라인이라는 말의 뜻이 손에 잡힙니다. 한 장에서 통했던 추출과 채점을 반복문에 넣는 순간, 사람의 개입 없이 여러 장이 같은 절차를 통과하게 됩니다. 다섯 장이 백 장이 되어도 코드는 목록만 길어질 뿐 구조가 같습니다. 수작업과의 차이는 속도보다 균일함에 있습니다. 사람은 백 장째쯤 지쳐서 기준이 흔들리지만, 파이프라인은 첫 장과 백 장째를 같은 기준으로 처리합니다. 대신 잘못된 기준도 똑같이 균일하게 적용되므로, 기준을 검증하는 단계가 그만큼 중요해집니다. 이 다섯 장에서는 항목 판정이 모두 일치했습니다. 정확도 100%입니다. 여기서 결론을 내리면 안 됩니다. 다섯 장은 너무 적습니다. 앞 실습에서 60개 표본으로도 정확도의 신뢰구간이 넓다고 했는데, 다섯 장이면 더합니다. 100%가 나왔다는 것은 "이 다섯 장에서는 틀린 데가 없었다"는 사실을 말할 뿐, 다음 100장에서도 그럴 것이라는 근거는 되지 못합니다. 그래서 더 큰 규모로 확인한 결과를 이어서 보겠습니다.

### 상담 녹음의 화자 구분 전사(STT)

**단계 5. 상담 녹음 전사**

**[3-43] 자연어 지시: 상담 녹음 화자 구분 전사**

```
이 상담 녹음을 한국어로 전사해줘. 발화자를 '상담원:'과 '고객:'으로 구분해서 줄 단위로.
```

**[3-44] 상담 녹음 전사**

In [ ]:
import os, io, json, wave, urllib.request

# 준비 칸이 상담 녹음을 만들어 둡니다. 음성 합성이 막혀 없을 때만 아래가 다시 만듭니다.
# 직접 녹음한 파일을 쓰시려면 audio/call1_engine_start.wav 로 올리고
# audio/call1_engine_start.txt 를 그 내용에 맞게 고쳐 두시면 이 부분은 건너뜁니다.
WAV, TXT = "audio/call1_engine_start.wav", "audio/call1_engine_start.txt"
FALLBACK = [
    ("상담원", "네, 고객센터입니다. 무엇을 도와드릴까요?"),
    ("고객", "굴착기 시동이 안 걸려서요. 아침부터 계속 그럽니다."),
    ("상담원", "장비 번호를 알려 주시겠어요?"),
    ("고객", "이엑스 이공오번입니다."),
    ("상담원", "계기판에 경고등이 들어와 있습니까?"),
    ("고객", "배터리 표시등이 깜빡입니다."),
    ("상담원", "어제 작업 마치고 시동을 껐을 때 이상은 없었습니까?"),
    ("고객", "없었습니다. 오늘 아침에만 그렇습니다."),
    ("상담원", "배터리 방전으로 보입니다. 단자 상태를 먼저 확인해 주시고, 정비 기사 배정해 드리겠습니다."),
    ("고객", "네, 부탁드립니다."),
]

def _make_call_wav():
    os.makedirs("audio", exist_ok=True)
    if os.path.exists(TXT):                       # 준비 칸이 남긴 대본을 그대로 씁니다
        script = [(l.split(":", 1)[0].strip(), l.split(":", 1)[1].strip())
                  for l in open(TXT, encoding="utf-8").read().splitlines() if ":" in l]
    else:
        script = FALLBACK
        open(TXT, "w", encoding="utf-8").write(
            "\n".join(f"{who}: {line}" for who, line in script) + "\n")
    if PROVIDER != "openai":
        raise SystemExit("상담 녹음이 없습니다 — 휴대폰으로 30초쯤 대화를 녹음해 "
                         + WAV + " 로 올린 뒤, " + TXT + " 를 그 내용에 맞게 "
                         "고치고 다시 실행하세요.")
    voice = {"상담원": "nova", "고객": "onyx"}      # 두 사람으로 들리게 나눕니다
    print("상담 녹음을 만드는 중 …", end=" ", flush=True)
    chunks = []
    for i, (who, line) in enumerate(script, 1):
        body = json.dumps({"model": "tts-1", "voice": voice.get(who, "nova"),
                           "input": line, "response_format": "wav"}).encode()
        req = urllib.request.Request("https://api.openai.com/v1/audio/speech", data=body,
                                     headers={"Authorization": "Bearer " + API_KEY,
                                              "Content-Type": "application/json"})
        chunks.append(urllib.request.urlopen(req, timeout=120).read())
        print(i, end=" ", flush=True)
    with wave.open(io.BytesIO(chunks[0])) as w0:
        ch, sw, sr = w0.getnchannels(), w0.getsampwidth(), w0.getframerate()
    gap = b"\x00" * int(sr * 0.35) * sw * ch          # 말 사이의 짧은 쉼
    with wave.open(WAV, "wb") as out:
        out.setnchannels(ch); out.setsampwidth(sw); out.setframerate(sr)
        for i, c in enumerate(chunks):
            with wave.open(io.BytesIO(c)) as w:
                if i: out.writeframes(gap)
                out.writeframes(w.readframes(w.getnframes()))
    with wave.open(WAV) as w:
        print(f"\n{WAV} 완성 — {w.getnframes()/w.getframerate():.1f}초 (대본 {len(script)}줄)")

if not os.path.exists(WAV):
    _make_call_wav()

transcript = gemini('''이 음성은 건설장비 고객센터 상담 녹음입니다. 전체를 한국어로 전사하세요.
발화자를 '상담원:'과 '고객:'으로 구분해 줄 단위로 적으세요. 전사 텍스트만 출력하세요.''',
                    "audio/call1_engine_start.wav", "audio/wav")
print(transcript[:400], "'''")

실행하면 상담 내용이 줄 단위로 전사되어 출력됩니다. 각 줄 앞에 상담원: 또는 고객:이 붙어 누가 한 말인지 구분됩니다. 화자 구분은 정확하게 이루어졌습니다. 음성을 글로 바꾸는 이 작업을 음성-텍스트 변환(Speech-to-Text, STT) 또는 전사라고 부릅니다. 예전에는 전사 전용 엔진을 따로 붙이고, 화자 구분은 또 다른 처리로 해결해야 했던 작업입니다. 멀티모달 모델은 소리를 알아듣는 일과 문맥을 이해하는 일을 함께 수행하므로, "상담원과 고객으로 구분해 적으라"는 자연어 지시 하나로 두 작업이 한 번에 처리됩니다. 목소리의 차이뿐 아니라 발화의 내용(안내하는 쪽과 증상을 호소하는 쪽)까지 근거로 화자를 가릴 수 있다는 것이 이 방식의 강점입니다. 화자를 나누라고 지시한 것에는 실무적 이유가 있습니다. 상담 기록을 나중에 활용할 때 필요한 정보는 대부분 고객의 발화에 들어 있습니다. 증상 설명, 장비 상태, 발생 시점 같은 것들입니다. 상담원의 발화는 안내와 확인이 대부분입니다. 화자가 구분되어 있어야 필요한 쪽만 골라 분석할 수 있습니다. 음성을 보내는 방식은 앞서 사진을 보낼 때와 같습니다. 파일을 요청 본문에 직접 실어 보냅니다. 여기에는 요청 하나가 20MB를 넘을 수 없다는 제약이 따릅니다. 몇 분짜리 상담 녹음은 문제가 없지만 한 시간짜리 회의 녹음은 이 방식으로 보낼 수 없습니다. 긴 녹음은 파일을 먼저 올려 두고 참조하는 방식을 쓰거나, 구간을 나눠 여러 번 호출한 뒤 이어 붙여야 합니다.

**[3-45] 문자 오차율(CER) 계산**

전사가 잘 되었는지 어떻게 확인할까요. 점검표에는 정답 JSON이 있었지만 전사에는 무엇을 정답으로 삼아야 할까요. 이 녹음에는 대본이 함께 있습니다. 학습용으로 만든 상담이라 대본을 먼저 쓰고 녹음했기 때문입니다. 대본이 곧 정답지입니다.

In [ ]:
import re
def norm(t):
    t = re.sub(r"^(상담원|고객)\s*:", "", t, flags=re.M)     # 화자 표시 제거
    return re.sub(r"[\s.,?!~'\"()\-''']", "", t)              # 공백·문장부호 제거
ref = norm(open("audio/call1_engine_start.txt", encoding="utf-8").read())
hyp = norm(transcript)
prev = list(range(len(hyp) + 1))                               # 편집거리(DP)
for i, rc in enumerate(ref, 1):
    cur = [i] + [0] * len(hyp)
    for j, hc in enumerate(hyp, 1):
        cur[j] = min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (rc != hc))
    prev = cur
print(f"정답 {len(ref)}자, 편집거리 {prev[-1]}, CER = {prev[-1]/len(ref):.2%}")

문자 오차율(Character Error Rate, CER)은 전사 결과를 정답으로 되돌리기 위해 고쳐야 하는 글자 수를 정답 길이로 나눈 값입니다. 고칠 것이 없으면 0이고, 값이 클수록 나쁩니다. 고쳐야 하는 글자 수를 세는 계산이 편집거리입니다. 글자를 하나 바꾸거나, 빠진 글자를 넣거나, 남는 글자를 지우는 세 가지 연산의 최소 횟수를 구합니다. 코드에서 이중 반복문이 하는 일이 그것입니다. !

### 도면 부품표(BOM) 추출 및 데이터베이스 적재

**[3-46] 도면 확인**

마지막 대상은 도면입니다. 도면 이미지 한 장에서 표제란과 부품표를 뽑아내고, 이번에는 데이터베이스에 넣는 데까지 진행합니다.

In [ ]:
display(Image("drawings/drawing_R-DWG-001.png", width=640))

화면에 도면이 나타납니다. 오른쪽 아래에 표제란이 있어 도면번호·도면명·적용기종·일자가 적혀 있고, 그 위에 부품표가 표 형태로 들어 있습니다. 부품표(Bill of Materials, BOM)는 하나의 제품이나 조립품에 들어가는 부품의 목록입니다. 부품번호·명칭·수량이 행으로 나열됩니다. 부품표와 짝을 이루는 개념이 부품 마스터입니다. 부품표가 조립품 하나의 구성 목록이라면, 부품 마스터는 회사가 다루는 전체 부품의 기준 목록으로, 발주·재고·단가 관리가 모두 이 목록의 부품번호를 기준으로 돌아갑니다. 도면의 부품표에는 있는데 마스터에는 없는 부품이 생기면, 그 부품은 도면상으로는 존재하지만 회사 시스템으로는 관리할 수 없는 상태가 됩니다. 이 실습의 마지막 단계에서 두 목록을 대조하는 이유를 미리 밝혀 두는 것입니다.

**[3-47] 자연어 지시: 도면 표제란과 BOM 추출**

```
이 도면에서 표제란(도면번호·적용기종·일자)과 부품표(BOM)를 JSON으로 추출하고
SQLite 테이블에 넣어줘.
```

**[3-48] 도면에서 표제란과 BOM 추출**

In [ ]:
dwg = as_json(gemini('''이 도면에서 다음을 JSON으로 추출하세요.
{"도면번호": "", "도면명": "", "적용기종": "", "일자": "",
 "BOM": [{"부품번호": "", "명칭": "", "수량": ""}]}
부품번호가 없는 행은 부품번호를 "(품번 미등록)"으로 적으세요. JSON만 출력하세요.''',
                    "drawings/drawing_R-DWG-001.png", "image/png"))
print(json.dumps(dwg, ensure_ascii=False, indent=1))

실행하면 표제란 정보와 부품표가 담긴 JSON이 출력됩니다. 최상위에 도면번호·도면명·적용기종·일자가 있고, BOM 아래에 부품 행이 배열로 들어갑니다. 이 도면의 부품표는 3행입니다. 프롬프트의 마지막 지시에 주목하기 바랍니다. 부품번호가 없는 행은 (품번 미등록)으로 적으라고 했습니다. 앞의 점검표 채점에서 확인한 빈칸 문제에 대한 대응입니다. 지시를 주지 않으면 세 가지 중 하나가 일어납니다. 그 행을 아예 빠뜨리거나, 부품번호를 빈 문자열로 두거나, 비슷한 품번을 만들어 넣습니다. 셋 다 곤란합니다. 행을 빠뜨리면 부품 하나가 통째로 사라지고, 빈 문자열은 나중에 다른 결측과 구분되지 않으며, 지어낸 품번은 가장 위험합니다. (품번 미등록)이라는 명시적인 표시를 쓰면 "이 행에는 부품번호가 원래 없었다"는 사실이 데이터에 남습니다. 공백을 지우지 않고 공백으로 기록하는 것입니다. 이 한 줄이 뒤에서 관리 공백을 찾아내는 근거가 됩니다.

**[3-49] SQLite에 도면 ·BOM 적재**

도면 이미지에서 표제란 정보와 부품표(BOM)를 추출하여 데이터베이스에 저장하는 과정을 진행합니다. 실행하면 데이터베이스에 저장된 부품표 행이 차례로 출력됩니다. 3행이 그대로 들어간 것을 확인할 수 있습니다. 여기서 쓴 SQLite부터 소개하겠습니다. 데이터베이스라고 하면 별도의 서버를 설치하고 전담자가 운영하는 큰 시스템을 떠올리기 쉽지만, SQLite는 파일 하나로 동작하는 가벼운 데이터베이스입니다. 설치할 프로그램도 접속할 서버도 없이 파이썬에 기본으로 포함된 기능만으로 만들고 읽을 수 있어, 실습과 소규모 업무 도구에 알맞습니다. 지금 만든 factory_docs.db가 그 파일이고, 다른 컴퓨터로 복사해 가도 그대로 열립니다. 조회에 쓰는 언어가 SQL(Structured Query Language)이며, "bom 테이블에서 모든 행을 가져와라" 같은 요구를 정해진 문법으로 적는 방식입니다. 코드의 SELECT * FROM bom이 바로 그 문장입니다. 테이블을 둘로 나눈 설계를 짚어 두겠습니다. drawings는 도면 한 건에 한 행이고, bom은 부품 한 건에 한 행입니다. 도면 하나에 부품이 여럿 딸리는 관계이므로 한 표에 담으면 도면 정보가 부품 수만큼 반복됩니다. 반복이 생기면 나중에 도면명을 고칠 때 여러 행을 함께 고쳐야 하고, 하나라도 빠뜨리면 데이터가 어긋납니다.

In [ ]:
import sqlite3
con = sqlite3.connect("factory_docs.db")
con.execute("CREATE TABLE IF NOT EXISTS drawings(도면번호 TEXT PRIMARY KEY, 도면명 TEXT, 적용기종 TEXT, 일자 TEXT)")
con.execute("CREATE TABLE IF NOT EXISTS bom(도면번호 TEXT, 부품번호 TEXT, 명칭 TEXT, 수량 TEXT)")
con.execute("INSERT OR REPLACE INTO drawings VALUES(?,?,?,?)",
            (dwg["도면번호"], dwg["도면명"], dwg["적용기종"], dwg["일자"]))
con.execute("DELETE FROM bom WHERE 도면번호=?", (dwg["도면번호"],))
con.executemany("INSERT INTO bom VALUES(?,?,?,?)",
                [(dwg["도면번호"], r["부품번호"], r["명칭"], r["수량"]) for r in dwg["BOM"]])
con.commit()
print("적재된 BOM:")
for row in con.execute("SELECT * FROM bom"):
    print(" ", row)

추출한 부품번호가 회사의 부품 마스터에 실제로 있는지 확인합니다.

**[3-50] 부품 마스터와 대조**

In [ ]:
import pandas as pd
master = pd.read_csv("parts/parts_master.csv")
bom = pd.read_sql("SELECT * FROM bom", con)
bom["마스터등록"] = bom["부품번호"].isin(master["부품번호"])
bom[["부품번호", "명칭", "수량", "마스터등록"]]

부품표 3행에 마스터 등록 여부가 붙은 표가 출력됩니다. 결과는 등록 2건, 미등록 1건이었습니다. 도면에는 있는데 부품 마스터에는 없는 부품이 하나 나왔습니다. 이 한 줄이 오늘 실습의 도착점입니다. 미등록 부품이 있다는 것은 그 부품을 발주하거나 재고를 관리할 근거가 회사 시스템에 없다는 뜻입니다. 정비사가 그 부품이 필요해도 시스템에서 찾을 수 없고, 결국 전화로 문의하거나 도면을 직접 열어 확인하게 됩니다. 시간이 걸리는 것은 물론이고 기록도 남지 않습니다. 이 공백이 오늘 처음 생긴 것은 아닙니다. 도면과 마스터가 각각 다른 폴더에 파일로 있는 동안 계속 있었고, 아무도 몰랐을 뿐입니다. 대조가 가능해지는 순간 드러난 것입니다. 문서를 데이터로 바꾸면 그동안 보이지 않던 관리 공백이 드러납니다. 발견한 공백을 마스터에 등록하는 것까지가 한 사이클입니다. 도면 100장에 같은 파이프라인을 적용하면 미등록 부품 목록이 한 번에 나오고, 그 목록이 곧 부품 마스터 정비 계획서가 됩니다. 이 과정 1일차에 데이터 인벤토리를 만들며 "무엇이 어디에 있는지" 를 파악했다면, 여기서는 "무엇이 없는지"를 찾아낸 셈입니다.

## 7. 심화 실습: 자연어 질의 기반 데이터베이스 조회

### 실습 안내

**[3-51] 프롬프트: 자연어 질문을 SQL로**

```
당신은 SQLite 데이터베이스를 다루는 분석가입니다.
아래는 이 데이터베이스의 표 구조입니다.
(여기에 표 구조를 붙여 넣습니다)
다음 질문에 답하는 SELECT 문 하나만 출력하세요.
질문: (여기에 현장의 질문을 적습니다)
규칙
- SELECT 문만 출력하고 설명이나 코드 표시는 붙이지 마세요.
- 위 구조에 없는 표나 열은 쓰지 마세요.
- 데이터를 바꾸는 문장(INSERT, UPDATE, DELETE, DROP)은 쓰지 마세요.
- 날짜 비교가 필요하면 저장된 형식에 맞춰 쓰세요.
```

---

## 마무리 — 오늘 코드를 파일로 받아 가기

아래 칸을 실행하면 **오늘 내가 실행한 코드 전부**가 `.py` 파일 하나로 내려받아집니다.
내가 고쳐 쓴 내용도 그대로 담기니, 회사에 돌아가 그대로 다시 돌려 볼 수 있습니다.

노트북 자체를 남기려면 「파일 → 드라이브에 사본 저장」도 함께 해 두세요.
막히는 곳은 학습사이트의 같은 절을 함께 보면 설명이 있습니다.

In [ ]:
# 오늘 실행한 코드를 파이썬 파일(.py) 한 장으로 내려받습니다.
# 이 칸은 실습을 다 끝낸 뒤 마지막에 한 번만 실행하세요.
import datetime
NL = chr(10)

runs = []
for n, cell in enumerate(In[1:], start=1):          # In = 지금까지 실행한 칸들
    if "files.download" in cell:                    # 이 칸 자신은 뺍니다
        continue
    keep = [ln for ln in cell.splitlines()          # 코랩 전용 명령(!apt-get 등)은 줄 단위로 뺍니다
            if not ln.lstrip().startswith(("!", "%"))]
    t = NL.join(keep).strip()
    if not t:                                       # 빈 칸도 뺍니다
        continue
    runs.append("# ─────────── 실행 " + str(n) + " ───────────" + NL + t + NL)

path = "DAY3_실습_내코드.py"
head = ("# DAY 3 실습 — 내가 실행한 코드 모음" + NL
        + f"# 저장 시각 {datetime.datetime.now():%Y-%m-%d %H:%M}" + NL
        + "# 코랩에서 실행한 순서 그대로입니다. 내가 고친 내용도 그대로 담깁니다." + NL + NL)
with open(path, "w", encoding="utf-8") as f:
    f.write(head + NL.join(runs))
print(path + " 저장 — 칸 " + str(len(runs)) + "개")

try:
    from google.colab import files
    files.download(path)                            # 내 PC로 내려받기
except ImportError:
    print("코랩이 아니면 왼쪽 폴더 아이콘에서 직접 내려받으세요.")